# Alakazam / Dudunsparce experiment

Attach **the-pokemon-company-ptcg-ai-battle-challenge-playground** in Kaggle's
Add Input panel, then Run All. No pip installs or network access are required.
The notebook contains the Python agent and runner, locates the attached native
SDK, tests against random play and writes a `submission.tar.gz` artifact.

Run the agent-file setup cell before evaluation. It writes every required
module together; evaluation and packaging refresh those files automatically.

Generated from the project's tested Python modules with
`python -X utf8 tools/build_notebook.py`. Change the Python sources and regenerate
to keep the notebook synchronized. Saved outputs are intentionally cleared.

The benchmark uses independent single games with balanced seats and starting
order, not Kaggle BO3 matches. `POLICY_SEED` does not seed the engine's deals or
coin flips. This is a heuristic baseline, not a trained model.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import tarfile
from datetime import datetime, timezone

GAMES = 20              # Use 100 or more for a larger evaluation.
AGENT = "alakazam"
OPPONENT = "abomasnow"  # random uses the candidate deck; named pilots use their own.
POLICY_SEED = 20261017
SDK_ROOT = None         # Optional explicit path to the folder containing cg/.

local_root = Path.cwd().resolve()
if local_root.name == "ipynb":
    local_root = local_root.parent
on_kaggle = Path("/kaggle/working").is_dir()
WORK_DIR = Path("/kaggle/working/alakazam") if on_kaggle else local_root / "ipynb/work/alakazam"
WORK_DIR.mkdir(parents=True, exist_ok=True)

if SDK_ROOT is None:
    input_root = Path("/kaggle/input") if on_kaggle else local_root / "the-pokemon-company-ptcg-ai-battle-challenge-playground"
    candidates = sorted({p.parent.parent for p in input_root.rglob("cg/api.py")})
    if len(candidates) != 1:
        attached = sorted(p.name for p in input_root.iterdir()) if input_root.is_dir() else []
        raise RuntimeError(
            f"Expected one supplied cg SDK under {input_root}; found {candidates}. "
            f"Visible input entries: {attached or '<none>'}. "
            "In Kaggle, use Add Input > Competition Datasets to attach "
            "The Pokemon Company - PTCG AI Battle Challenge Playground. "
            "If setting SDK_ROOT explicitly, use the folder containing cg/api.py.")
    SDK_ROOT = candidates[0]
SDK_ROOT = Path(SDK_ROOT)
if not (SDK_ROOT / "cg/api.py").is_file():
    raise FileNotFoundError(f"No cg/api.py under {SDK_ROOT}")
shutil.copytree(SDK_ROOT / "cg", WORK_DIR / "cg", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
print(f"Workspace: {WORK_DIR}\nSDK: {SDK_ROOT}")


In [ ]:
# Set up all agent files together; run this cell before evaluation.
PROJECT_SOURCES = {
    'pokemon_agent/__init__.py': r'''"""Local Pokémon TCG agents and evaluation tools."""
''',
    'pokemon_agent/sdk.py': r'''"""Locate the supplied SDK without changing the downloaded competition files."""
from functools import lru_cache
import importlib
from pathlib import Path
import sys

ROOT = Path(__file__).resolve().parents[1]
SAMPLE = (ROOT / "the-pokemon-company-ptcg-ai-battle-challenge-playground"
          / "sample_submission/sample_submission/sample_submission")
DEFAULT_DECK = ROOT / "deck.csv"


def read_deck(path=DEFAULT_DECK):
    """Read exactly 60 integer card IDs; native battle_start checks legality."""
    rows = Path(path).read_text(encoding="utf-8-sig").splitlines()
    if len(rows) != 60 or any(not row.strip() for row in rows):
        raise ValueError(f"{path}: expected exactly 60 nonempty card-ID rows")
    try:
        return [int(row) for row in rows]
    except ValueError as error:
        raise ValueError(f"{path}: every row must be an integer card ID") from error


@lru_cache(maxsize=1)
def load_sdk():
    """Prefer a packaged cg/ beside main.py, otherwise use the supplied sample."""
    base = ROOT if (ROOT / "cg/api.py").is_file() else SAMPLE
    if not (base / "cg/api.py").is_file():
        raise FileNotFoundError(f"Supplied SDK not found under {base}")
    # Do not put the vendor directory ahead of this project's main.py.
    # Both the sample and our agent use that filename.
    if str(base) not in sys.path:
        sys.path.append(str(base))
    return importlib.import_module("cg.api"), importlib.import_module("cg.game")


@lru_cache(maxsize=1)
def metadata():
    api, _ = load_sdk()
    return ({c.cardId: c for c in api.all_card_data()},
            {a.attackId: a for a in api.all_attack()})


def validate_selection(observation, action):
    """Reject invalid policies instead of silently repairing their output."""
    select = observation["select"]
    if not isinstance(action, list) or any(type(i) is not int for i in action):
        raise ValueError("Policy must return list[int] (booleans are not indices)")
    if not select["minCount"] <= len(action) <= select["maxCount"]:
        raise ValueError("Selection count is outside minCount/maxCount")
    if len(set(action)) != len(action):
        raise ValueError("Duplicate option indices")
    if any(i < 0 or i >= len(select["option"]) for i in action):
        raise ValueError("Option index out of range")
''',
    'pokemon_agent/policy.py': r'''"""Deck-specific heuristic baseline. Uses visible observations, not search state.

Scores are preferences, not a complete damage/rules simulator. The engine supplies
the legal actions. No training or opponent-hand access is involved.
"""
from collections import Counter
from math import comb
import random

from .sdk import load_sdk, metadata, read_deck, validate_selection

WATER, KYOGRE, SNOVER, ABOMASNOW = 3, 721, 722, 723
SIGNAL, BELT, CYRANO, LILLIE, WAITRESS = 1145, 1158, 1205, 1227, 1235


class RandomAgent:
    """The supplied baseline's algorithm: sample exactly maxCount options."""

    def __init__(self, deck=None, seed=0):
        self.deck = list(deck) if deck is not None else read_deck()
        self.rng = random.Random(seed)

    def __call__(self, obs):
        if obs.get("select") is None:
            return list(self.deck)
        select = obs["select"]
        return self.rng.sample(range(len(select["option"])), select["maxCount"])


class AbomasnowAgent:
    """Set up Abomasnow, preserve useful resources and recycle with Kyogre."""

    def __init__(self, deck=None):
        self.deck = list(deck) if deck is not None else read_deck()
        self.cards, self.attacks = metadata()
        api, _ = load_sdk()
        self.O, self.C, self.A = api.OptionType, api.SelectContext, api.AreaType

    def __call__(self, obs):
        if obs.get("select") is None:
            return list(self.deck)
        select = obs["select"]
        if not select["option"]:
            action = []
        elif select["context"] == self.C.MAIN:
            action = [max(range(len(select["option"])),
                          key=lambda i: self.main_score(obs, select["option"][i]))]
        else:
            action = self.select_effect(obs)
        validate_selection(obs, action)
        return action

    @staticmethod
    def players(obs):
        state = obs["current"]
        return state["players"][state["yourIndex"]], state["players"][1 - state["yourIndex"]]

    @staticmethod
    def field(player):
        return [p for p in player["active"] + player["bench"] if p is not None]

    def resolve(self, obs, option):
        """Resolve only cards exposed in this observation; facedown stays None."""
        state = obs["current"]
        player = state["players"][option.get("playerIndex", state["yourIndex"])]
        area = option.get("area", self.A.HAND if option["type"] == self.O.PLAY else None)
        index = option.get("index")
        areas = {self.A.HAND: player["hand"], self.A.DISCARD: player["discard"],
                 self.A.ACTIVE: player["active"], self.A.BENCH: player["bench"],
                 self.A.PRIZE: player["prize"], self.A.DECK: obs["select"].get("deck"),
                 self.A.LOOKING: state.get("looking"), self.A.STADIUM: state["stadium"]}
        values = areas.get(area)
        if values is None or index is None or not 0 <= index < len(values):
            return None
        card = values[index]
        if card and option["type"] in (self.O.ENERGY_CARD, self.O.ENERGY):
            return card["energyCards"][option["energyIndex"]]
        if card and option["type"] == self.O.TOOL_CARD:
            return card["tools"][option["toolIndex"]]
        return card

    def target(self, obs, option):
        me, _ = self.players(obs)
        area = option.get("inPlayArea")
        group = me["active"] if area == self.A.ACTIVE else me["bench"]
        index = option.get("inPlayIndex")
        return group[index] if index is not None and index < len(group) else None

    def water_distribution(self, obs):
        """Marginal draw distribution over unknown deck/prizes, not true order."""
        me, _ = self.players(obs)
        visible = list(me["hand"] or []) + me["discard"] + [c for c in me["prize"] if c]
        for p in self.field(me):
            visible += [p] + p["energyCards"] + p["tools"] + p["preEvolution"]
        unique = {c["serial"]: c for c in visible}
        unknown = me["deckCount"] + sum(c is None for c in me["prize"])
        water = max(0, min(unknown, self.deck.count(WATER) - sum(c["id"] == WATER for c in unique.values())))
        draws = min(6, me["deckCount"])
        if unknown == 0 or draws == 0:
            return [(0, 1.0)]
        denominator = comb(unknown, draws)
        return [(k, comb(water, k) * comb(unknown - water, draws - k) / denominator)
                for k in range(draws + 1) if k <= water and draws - k <= unknown - water]

    def attack_score(self, obs, attack_id, attacker=None):
        me, opponent = self.players(obs)
        attacker = attacker or next(iter(me["active"]), None)
        defender = next(iter(opponent["active"]), None)
        if not attacker or not defender:
            return -10.0
        attack = self.attacks[attack_id]
        if attack.name == "Hammer-lanche":
            distribution = [(100 * k, p) for k, p in self.water_distribution(obs)]
        elif attack.name == "Riptide":
            distribution = [(20 * sum(c["id"] == WATER for c in me["discard"]), 1.0)]
        else:
            distribution = [(max(0, attack.damage), 1.0)]
        target_info, attacker_info = self.cards[defender["id"]], self.cards[attacker["id"]]
        # This deck cannot disable Sylveon's Safeguard or Crustle's Mysterious
        # Rock Inn. Both prevent attack damage from ex (including Mega ex).
        # Frost Barrier can still apply its defensive effect; Hammer-lanche
        # would only spend deck cards. A non-ex Kyogre remains a valid attacker.
        if defender["id"] in (330, 345) and (attacker_info.ex or attacker_info.megaEx):
            return 20.0 if attack.name == "Frost Barrier" else -1.0
        prizes = 3 if target_info.megaEx else 2 if target_info.ex else 1
        belt = any(t["id"] == BELT for t in attacker["tools"]) and (target_info.ex or target_info.megaEx)
        expected, knockout = 0.0, 0.0
        for damage, probability in distribution:
            if damage > 0:
                damage += 50 if belt else 0
                if target_info.weakness == attacker_info.energyType:
                    damage *= 2
                if target_info.resistance == attacker_info.energyType:
                    damage = max(0, damage - 30)
            expected += min(damage, defender["hp"]) * probability
            knockout += probability * (damage >= defender["hp"])
        winning_ko = prizes >= len(me["prize"]) or len(self.field(opponent)) == 1
        score = 200 + expected * 0.4 + knockout * (350 * prizes + (5000 if winning_ko else 0))
        if attack.name == "Hammer-lanche":
            score -= 25  # Prefer a sufficient Frost Barrier over needless milling.
            if me["deckCount"] <= 6:
                score -= 1800 * (1 - (knockout if winning_ko else 0))
        elif attack.name == "Frost Barrier":
            score += 20
        elif attack.name == "Riptide":
            discarded = sum(c["id"] == WATER for c in me["discard"])
            score += min(discarded, 12) * (15 if me["deckCount"] <= 8 else 2)
        elif attack.name == "Swirling Waves":
            score -= 45
        return score if expected > 0 else -1

    def can_pay(self, pokemon, attack_id):
        available = Counter(pokemon["energies"])
        costs = self.attacks[attack_id].energies
        # This fixed deck contains only Basic Water; no special-energy semantics.
        if len(pokemon["energies"]) < len(costs):
            return False
        return all(available[t] >= count for t, count in Counter(costs).items() if t != 0)

    def ready_score(self, obs, pokemon):
        scores = [self.attack_score(obs, aid, pokemon) for aid in self.cards[pokemon["id"]].attacks
                  if self.can_pay(pokemon, aid)]
        return max(scores, default=-10)

    def energy_priority(self, obs, pokemon):
        if not pokemon:
            return -100
        me, opponent = self.players(obs)
        active = bool(me["active"] and pokemon["serial"] == me["active"][0]["serial"])
        count = len(pokemon["energies"])
        if pokemon["id"] == KYOGRE and not active and count == 0 and me["active"]:
            front = me["active"][0]
            # Powerful Hand places counters: Frost Barrier cannot reduce it.
            # Public handCount estimates the threat; no hidden cards are read.
            counter_threat = 20 * opponent["handCount"] if any(
                p["id"] == 743 and p["energies"] for p in self.field(opponent)) else 0
            if (front["id"] == ABOMASNOW and len(front["energies"]) >= 2
                    and counter_threat >= front["hp"]):
                # Do not divert the attachment that enables Hammer-lanche now.
                return 130  # Prepare a one-Prize Riptide attacker before the KO.
        if pokemon["id"] in (SNOVER, ABOMASNOW):
            needed = 2 if count < 2 else 3
            score = 90 if count < 2 else 35 if count == 2 else -20
            if (active and pokemon["id"] == ABOMASNOW and count == 3 and me["deckCount"] <= 8
                    and any(p["id"] == KYOGRE and len(p["energies"]) >= 1 for p in me["bench"])):
                needed, score = 4, 110  # Pay retreat and recycle before decking out.
        else:
            # A lone Kyogre with an empty discard cannot deal damage with
            # Riptide. Charge Swirling Waves (or its three-Energy retreat).
            discarded = sum(c["id"] == WATER for c in me["discard"])
            needed = 3 if active and (discarded < 7 or any(p["id"] in (SNOVER, ABOMASNOW) for p in me["bench"])) else 1
            score = 60 if count == 0 else 45 if count < needed else -25
        return score + (35 if active and count < needed else 0) + min(pokemon["hp"], 100) / 100

    def main_score(self, obs, option):
        me, opponent = self.players(obs)
        kind = option["type"]
        hand = Counter(c["id"] for c in me["hand"])
        field = Counter(p["id"] for p in self.field(me))
        if kind == self.O.ATTACK:
            return self.attack_score(obs, option["attackId"])
        if kind == self.O.END:
            return 0
        if kind == self.O.EVOLVE:
            target = self.target(obs, option)
            if (option.get("inPlayArea") == self.A.BENCH and len(opponent["prize"]) <= 3
                    and len(target["energies"]) < 2):
                # A gust can turn an idle Mega into the opponent's final Prizes.
                # Attach to Snover first; reconsider evolution once it can attack.
                return -10
            return 1100 + self.energy_priority(obs, target)
        if kind == self.O.ATTACH:
            card, target = self.resolve(obs, option), self.target(obs, option)
            if card["id"] == BELT:
                return 850 if target["id"] in (SNOVER, ABOMASNOW) else 5
            priority = self.energy_priority(obs, target)
            return 900 + priority if priority > 0 else -5
        if kind == self.O.RETREAT:
            if not me["bench"]:
                return -100
            available = [self.attack_score(obs, o["attackId"]) for o in obs["select"]["option"] if o["type"] == self.O.ATTACK]
            active_value = max(available, default=-10)
            best_bench = max(self.ready_score(obs, p) for p in me["bench"])
            return 1050 if best_bench > active_value + 150 else -100
        if kind == self.O.PLAY:
            cid = self.resolve(obs, option)["id"]
            if cid == SNOVER:
                return 1200 if field[SNOVER] + field[ABOMASNOW] < 2 else -10
            if cid == KYOGRE:
                return 1000 if field[KYOGRE] < 1 else -10
            need_evolution = field[SNOVER] > hand[ABOMASNOW]
            if cid == SIGNAL:
                return 950 if need_evolution else -10
            if cid == CYRANO:
                return 800 if need_evolution and not hand[SIGNAL] else -10
            if cid == WAITRESS:
                priority = max((self.energy_priority(obs, p) for p in self.field(me)), default=-100)
                return 800 + priority if priority > 0 else -10
            if cid == LILLIE:
                draws = 8 if len(me["prize"]) == 6 else 6
                if me["deckCount"] + len(me["hand"]) - 1 <= draws:
                    return -100
                missing_setup = field[SNOVER] + field[ABOMASNOW] < 2 or (need_evolution and not hand[SIGNAL])
                return 750 if missing_setup or len(me["hand"]) <= draws else 30
            return -10
        if kind == self.O.ABILITY:
            return 50
        return -20

    def card_value(self, obs, card, chosen):
        if card is None:  # A face-down prize has no known identity.
            return 1
        me, _ = self.players(obs)
        cid = card["id"]
        field = Counter(p["id"] for p in self.field(me))
        hand = Counter(c["id"] for c in me["hand"])
        if cid == ABOMASNOW:
            return 90 if hand[cid] + chosen[cid] < max(1, field[SNOVER]) else -2
        if cid == SNOVER:
            return 85 if field[cid] + field[ABOMASNOW] + chosen[cid] < 2 else -2
        if cid == KYOGRE:
            return 75 if field[cid] + chosen[cid] < 1 else -2
        if cid == WATER:
            return 30 if hand[cid] < 3 else 5
        return {LILLIE: 70, WAITRESS: 65, SIGNAL: 50, CYRANO: 40, BELT: 55}.get(cid, 1)

    def select_effect(self, obs):
        select = obs["select"]
        context, options = select["context"], select["option"]
        C, O = self.C, self.O
        me, _ = self.players(obs)
        chosen, indices = Counter(), []
        discard = context in (C.DISCARD, C.DISCARD_CARD_OR_ATTACHED_CARD, C.DISCARD_ENERGY_CARD,
                              C.DISCARD_ENERGY, C.DISCARD_TOOL_CARD)

        def score(i):
            option = options[i]
            kind = option["type"]
            if kind in (O.YES, O.NO):
                # Evolving early is valuable; the evaluation runner can control
                # first/second order independently of this policy preference.
                yes = context != C.MULLIGAN
                return 10 if (kind == O.YES) == yes else -1
            if kind == O.NUMBER:
                return option["number"] if context != C.DRAW_COUNT else min(option["number"], max(0, me["deckCount"] - 1)) - (0.01 * option["number"])
            if kind == O.ATTACK:
                return self.attack_score(obs, option["attackId"])
            card = self.resolve(obs, option)
            if context == C.SETUP_ACTIVE_POKEMON:
                return 100 if card and card["id"] == SNOVER else 60
            if context in (C.SWITCH, C.TO_ACTIVE) and card and "energies" in card:
                return self.ready_score(obs, card) + (30 if card["id"] == ABOMASNOW else 0)
            if context == C.ATTACH_FROM and card and "energies" in card:
                return self.energy_priority(obs, card)
            if context == C.ATTACH_TO:
                return 100 if card and card["id"] == WATER else 5
            value = self.card_value(obs, card, chosen)
            return -value if discard else value

        cap = select["maxCount"]
        # Discard only what is required. Energy prompts can ask again until the
        # cost is paid; obey their supplied bounds on every selection.
        if discard:
            cap = select["minCount"]
        for _ in range(cap):
            remaining = [i for i in range(len(options)) if i not in indices]
            best = max(remaining, key=score)
            if score(best) <= 0 and len(indices) >= select["minCount"]:
                break
            indices.append(best)
            card = self.resolve(obs, options[best])
            if card:
                chosen[card["id"]] += 1
        return indices
''',
    'pokemon_agent/alakazam.py': r'''"""One-Prize Alakazam/Dudunsparce pilot using only its legal observation.

Powerful Hand scales with hand size, not discarded Energy. The Abomasnow base
supplies selection validation and visible-card resolution; strategy is replaced.
"""
from collections import Counter

from .policy import AbomasnowAgent
from .sdk import ROOT, read_deck

ABRA, KADABRA, ALAKAZAM, DUNSPARCE, DUDUNSPARCE = 741, 742, 743, 65, 66
PSYCHIC, TELEPATH = 5, 19
POFFIN, CANDY, PAD, DAWN, HILDA = 1086, 1079, 1152, 1231, 1225
LILLIE, BOSS, LANA, STRETCHER, CAGE, RED_CARD = 1227, 1182, 1184, 1097, 1264, 1413
DECK_PATH = ROOT / "decks/alakazam.csv"


class AlakazamAgent(AbomasnowAgent):
    """Grow a hand through evolution/draw and trade one-Prize attackers."""

    def __init__(self, deck=None):
        # Basic Psychic and Telepath each supply one Psychic in engine energies;
        # the inherited typed-cost check therefore handles both correctly.
        super().__init__(read_deck(DECK_PATH) if deck is None else deck)

    def lines(self, player):
        counts = Counter(p["id"] for p in self.field(player))
        return counts[ABRA] + counts[KADABRA] + counts[ALAKAZAM], counts[DUNSPARCE] + counts[DUDUNSPARCE]

    def attack_score(self, obs, attack_id, attacker=None, defender=None, hand_count=None):
        me, opponent = self.players(obs)
        attacker = attacker or next(iter(me["active"]), None)
        defender = defender or next(iter(opponent["active"]), None)
        if not attacker or not defender:
            return -10
        attack = self.attacks[attack_id]
        if attack_id == 1072:
            damage = 20 * (len(me["hand"]) if hand_count is None else hand_count)
        else:
            damage = max(0, attack.damage)
            target = self.cards[defender["id"]]
            if damage and target.weakness == self.cards[attacker["id"]].energyType:
                damage *= 2
            if damage and target.resistance == self.cards[attacker["id"]].energyType:
                damage = max(0, damage - 30)
        target = self.cards[defender["id"]]
        prizes = 3 if target.megaEx else 2 if target.ex else 1
        winning = prizes >= len(me["prize"]) or len(self.field(opponent)) == 1
        if damage >= defender["hp"]:
            return 1800 * prizes + (6000 if winning else 0)
        return 200 + min(damage, defender["hp"]) * 0.4 if damage else -1

    def energy_priority(self, obs, pokemon):
        if not pokemon or pokemon["energies"] or pokemon["id"] not in (ABRA, KADABRA, ALAKAZAM):
            return -10
        me, _ = self.players(obs)
        active = bool(me["active"] and pokemon["serial"] == me["active"][0]["serial"])
        return {ALAKAZAM: 140, KADABRA: 100, ABRA: 90}.get(pokemon["id"], -10) + (20 if active else 0)

    def card_value(self, obs, card, chosen):
        if card is None:
            return 1
        me, _ = self.players(obs)
        cid = card["id"]
        hand = Counter(c["id"] for c in me["hand"])
        field = Counter(p["id"] for p in self.field(me))
        basics, draws = self.lines(me)
        if cid == ALAKAZAM:
            return 110 if hand[cid] + chosen[cid] < max(1, field[KADABRA] + field[ABRA]) else 15
        if cid == KADABRA:
            return 100 if hand[cid] + chosen[cid] < max(1, field[ABRA]) else 15
        if cid == ABRA:
            return 95 if basics + hand[cid] + chosen[cid] < 3 else 15
        if cid == DUNSPARCE:
            return 85 if draws + hand[cid] + chosen[cid] < 2 else 15
        if cid == DUDUNSPARCE:
            return 90 if hand[cid] + chosen[cid] < max(1, field[DUNSPARCE]) else 15
        if cid in (PSYCHIC, TELEPATH):
            have = hand[PSYCHIC] + hand[TELEPATH] + chosen[PSYCHIC] + chosen[TELEPATH]
            return (105 if have == 0 else 40) + (5 if cid == TELEPATH and basics < 3 else 0)
        return {CANDY: 80 if field[ABRA] and hand[ALAKAZAM] else 30,
                DAWN: 75, HILDA: 70, POFFIN: 65, PAD: 60, LANA: 50,
                LILLIE: 50 if len(me["hand"]) < 6 else 5}.get(cid, 20)

    def main_score(self, obs, option):
        me, opponent = self.players(obs)
        kind = option["type"]
        hand = Counter(c["id"] for c in me["hand"])
        basics, draws = self.lines(me)
        if kind == self.O.ATTACK:
            return self.attack_score(obs, option["attackId"])
        if kind == self.O.END:
            return 0
        if kind == self.O.EVOLVE:
            card, target = self.resolve(obs, option), self.target(obs, option)
            if card["id"] == DUDUNSPARCE and len(self.field(me)) == 1:
                return -10
            if me["deckCount"] <= 3:
                return 10 if card["id"] == ALAKAZAM else -10
            return (1250 if card["id"] == ALAKAZAM else 1100) + max(0, self.energy_priority(obs, target))
        if kind == self.O.ATTACH:
            card, target = self.resolve(obs, option), self.target(obs, option)
            priority = self.energy_priority(obs, target)
            return 1000 + priority + (25 if card["id"] == TELEPATH and basics < 3 else 0) if priority > 0 else -10
        if kind == self.O.RETREAT:
            active = max((self.attack_score(obs, o["attackId"]) for o in obs["select"]["option"]
                          if o["type"] == self.O.ATTACK), default=-10)
            bench = max((self.ready_score(obs, p) for p in me["bench"]), default=-10)
            return 1600 if bench > active + 50 else -10
        if kind == self.O.ABILITY:
            card = self.resolve(obs, option)
            if card and card["id"] == DUDUNSPARCE:
                return 950 if len(self.field(me)) > 1 and me["deckCount"] > 3 else -10
            return -10
        if kind != self.O.PLAY:
            return -10
        card = self.resolve(obs, option)
        cid = card["id"]
        if cid == ABRA:
            return 1200 if basics < 3 else -10
        if cid == DUNSPARCE:
            return 1000 if draws < 2 else -10
        if cid == CANDY:
            eligible = any(p["id"] == ABRA and not p["appearThisTurn"] for p in self.field(me))
            if not eligible or not hand[ALAKAZAM]:
                return -10
            return 1350 if not any(p["id"] == ALAKAZAM for p in self.field(me)) else 500
        if cid == POFFIN:
            return 1050 if basics < 2 or draws < 1 else 700 if basics < 3 or draws < 2 else -10
        if cid == PAD:
            return 900 if basics < 2 or draws < 1 or any(p["id"] in (ABRA, KADABRA) for p in self.field(me)) else -10
        if cid in (DAWN, HILDA):
            return (850 if cid == DAWN else 800) if me["deckCount"] > 4 else -10
        if cid == LILLIE:
            count = 8 if len(me["prize"]) == 6 else 6
            return 880 if len(me["hand"]) < count - 1 and me["deckCount"] > count else -10
        if cid == LANA:
            available = sum(self.cards[c["id"]].cardType in (0, 5) for c in me["discard"])
            return 860 if available >= 2 else -10
        if cid == STRETCHER:
            return 880 if any(self.card_value(obs, c, Counter()) >= 85 for c in me["discard"]) else -10
        if cid == CAGE:
            return 700 if any(p["id"] in (121, 132, 133) for p in self.field(opponent)) else -10
        if cid == RED_CARD:
            return 700 if opponent["handCount"] > 5 else -10
        if cid == BOSS:
            if not me["active"] or me["active"][0]["id"] != ALAKAZAM or not self.can_pay(me["active"][0], 1072):
                return -10
            current = self.attack_score(obs, 1072)
            best = max((self.attack_score(obs, 1072, defender=p, hand_count=len(me["hand"])-1)
                        for p in opponent["bench"]), default=-10)
            return best + 20 if best >= 1800 and best > current else -10
        return -10

    def select_effect(self, obs):
        select = obs["select"]
        context, options = select["context"], select["option"]
        me, opponent = self.players(obs)
        chosen, indices = Counter(), []
        discard = context in (self.C.DISCARD, self.C.DISCARD_CARD_OR_ATTACHED_CARD,
                              self.C.DISCARD_ENERGY_CARD, self.C.DISCARD_ENERGY, self.C.DISCARD_TOOL_CARD)

        def score(index):
            option = options[index]
            kind = option["type"]
            if kind in (self.O.YES, self.O.NO):
                yes = context != self.C.MULLIGAN
                if context == self.C.ACTIVATE:
                    source = select.get("contextCard") or {}
                    draw = {KADABRA: 2, ALAKAZAM: 3}.get(source.get("id"), 0)
                    yes = me["deckCount"] > draw
                return 10 if (kind == self.O.YES) == yes else -1
            if kind == self.O.NUMBER:
                return min(option["number"], max(0, me["deckCount"]-1)) - .01*option["number"]
            if kind == self.O.ATTACK:
                return self.attack_score(obs, option["attackId"])
            if kind == self.O.EVOLVE:
                target = self.target(obs, option)
                return 100 + (60 if target["energies"] else 0) + (30 if option["inPlayArea"] == self.A.ACTIVE else 0)
            card = self.resolve(obs, option)
            if context == self.C.SETUP_ACTIVE_POKEMON:
                return 100 if card and card["id"] == DUNSPARCE else 80
            if context in (self.C.SWITCH, self.C.TO_ACTIVE) and card and "energies" in card:
                if option.get("playerIndex", obs["current"]["yourIndex"]) != obs["current"]["yourIndex"]:
                    return self.attack_score(obs, 1072, defender=card)
                return self.ready_score(obs, card) + (40 if card["id"] == DUNSPARCE else 0)
            if context == self.C.ATTACH_FROM and card and "energies" in card:
                return self.energy_priority(obs, card)
            if context in (self.C.SETUP_BENCH_POKEMON, self.C.TO_BENCH, self.C.TO_FIELD) and card:
                basics, draws = self.lines(me)
                if card["id"] == ABRA:
                    return 100 if basics + chosen[ABRA] < 3 else -10
                if card["id"] == DUNSPARCE:
                    return 90 if draws + chosen[DUNSPARCE] < 2 else -10
            value = self.card_value(obs, card, chosen)
            return -value if discard else value

        cap = select["minCount"] if discard else select["maxCount"]
        for _ in range(cap):
            remaining = [i for i in range(len(options)) if i not in indices]
            best = max(remaining, key=score)
            if score(best) <= 0 and len(indices) >= select["minCount"]:
                break
            indices.append(best)
            card = self.resolve(obs, options[best])
            if card:
                chosen[card["id"]] += 1
        return indices
''',
    'pokemon_agent/agents.py': r'''"""Select a pilot and its matching deck for local comparisons."""
from .alakazam import AlakazamAgent
from .policy import AbomasnowAgent, RandomAgent
from .sdk import ROOT, DEFAULT_DECK, read_deck

AGENT_NAMES = ("abomasnow", "alakazam", "random")


def default_deck(name):
    if name == "alakazam":
        return read_deck(ROOT / "decks/alakazam.csv")
    if name not in AGENT_NAMES:
        raise ValueError(f"Unknown pilot: {name}")
    packaged = ROOT / "decks/abomasnow.csv"
    return read_deck(packaged if packaged.is_file() else DEFAULT_DECK)


def create_agent(name, deck=None, seed=0):
    deck = default_deck(name) if deck is None else deck
    if name == "abomasnow":
        return AbomasnowAgent(deck)
    if name == "alakazam":
        return AlakazamAgent(deck)
    if name == "random":
        return RandomAgent(deck, seed)
    raise ValueError(f"Unknown pilot: {name}")
''',
    'pokemon_agent/runner.py': r'''"""Run isolated local games and save results; no Kaggle service is contacted.

Example: python -m pokemon_agent.runner --games 100 --output results/baseline.json
The seed controls policy randomness only: the supplied native API seeds its own
shuffles/coins and does not expose a seed argument.
"""
import argparse
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import math
from pathlib import Path
import platform
import subprocess
import sys
import time
import traceback

from .sdk import ROOT, load_sdk, read_deck, validate_selection


def play_game(config, policies=None):
    """Play to an engine result, rejecting illegal output and bounding decisions."""
    from .agents import create_agent

    api, game = load_sdk()
    seat = config["agent_seat"]
    decks = [None, None]
    decks[seat], decks[1 - seat] = config["agent_deck"], config["opponent_deck"]
    if policies is None:
        policies = [None, None]
        for player, name in ((seat, config["agent"]), (1 - seat, config["opponent"])):
            policies[player] = create_agent(name, decks[player], config["policy_seed"] + player)
    result = {"game": config["game"], "agent_seat": seat, "policy_seed": config["policy_seed"],
              "scheduled_first_player": config["first_player"], "status": "error",
              "decisions": [0, 0], "policy_seconds": [0.0, 0.0],
              "max_policy_seconds": [0.0, 0.0]}
    counts = [Counter(), Counter()]
    observation, action, started, trace = None, None, False, None
    began = time.perf_counter()
    try:
        if config.get("trace_file"):
            path = Path(config["trace_file"])
            path.parent.mkdir(parents=True, exist_ok=True)
            trace = path.open("x", encoding="utf-8")
        observation, start = game.battle_start(*decks)
        started = bool(start.battlePtr)
        if observation is None:
            raise ValueError(f"Native deck rejection: player={start.errorPlayer}, type={start.errorType}")
        for step in range(config["max_decisions"]):
            state = observation["current"]
            result.update(turns=state["turn"], first_player=state["firstPlayer"],
                          native_round=state.get("round", 1))
            if state["result"] != -1:
                winner = state["result"]
                if winner not in (0, 1, 2):
                    raise ValueError(f"Unexpected native result {winner}")
                logs = [log for log in observation["logs"] if log["type"] == api.LogType.RESULT]
                result.update(status="completed", winner=winner,
                              outcome="draw" if winner == 2 else "win" if winner == seat else "loss",
                              finish_reason=logs[-1].get("reason") if logs else None)
                if trace:
                    trace.write(json.dumps({"event": "result", "winner": winner,
                                            "observation": {k: v for k, v in observation.items() if k != "search_begin_input"}},
                                           ensure_ascii=False) + "\n")
                break
            select = observation["select"]
            player = state["yourIndex"]
            before = time.perf_counter()
            if select["context"] == api.SelectContext.IS_FIRST:
                # Control first/second independently of seat for fair comparison.
                desired = api.OptionType.YES if player == config["first_player"] else api.OptionType.NO
                action = [next(i for i, option in enumerate(select["option"]) if option["type"] == desired)]
            else:
                action = policies[player](observation)
            elapsed = time.perf_counter() - before
            result["policy_seconds"][player] += elapsed
            result["max_policy_seconds"][player] = max(result["max_policy_seconds"][player], elapsed)
            result["decisions"][player] += 1
            if elapsed > config["decision_timeout"]:
                result["status"] = "timeout"
                raise TimeoutError(f"Player {player} decision exceeded {config['decision_timeout']} seconds")
            try:
                validate_selection(observation, action)
            except ValueError:
                result["status"] = "illegal_action"
                raise
            counts[player][api.SelectContext(select["context"]).name] += 1
            if trace:
                visible = {k: v for k, v in observation.items() if k != "search_begin_input"}
                trace.write(json.dumps({"step": step, "observation": visible, "action": action}, ensure_ascii=False) + "\n")
                trace.flush()
            try:
                observation = game.battle_select(action)
            except IndexError:
                result["status"] = "illegal_action"
                raise
        else:
            # A terminal result produced by the last allowed action still counts.
            state = observation["current"]
            if state["result"] in (0, 1, 2):
                winner = state["result"]
                result.update(status="completed", winner=winner, turns=state["turn"],
                              outcome="draw" if winner == 2 else "win" if winner == seat else "loss")
            else:
                result.update(status="decision_limit", error=f"Exceeded {config['max_decisions']} decisions")
    except Exception as error:
        result.update(error=f"{type(error).__name__}: {error}", traceback=traceback.format_exc())
        if observation:
            result["last_observation"] = {k: v for k, v in observation.items() if k != "search_begin_input"}
            result["last_action"] = action
    finally:
        if started:
            game.battle_finish()
        if trace:
            trace.close()
    result["wall_seconds"] = time.perf_counter() - began
    result["selection_contexts"] = [dict(c) for c in counts]
    return result


def run_isolated(config, timeout):
    """Terminate a hung/crashed native game without losing previous results."""
    try:
        worker = subprocess.run([sys.executable, "-X", "utf8", "-m", "pokemon_agent.runner", "--worker"],
                                input=json.dumps(config), capture_output=True, text=True,
                                encoding="utf-8", cwd=ROOT, timeout=timeout)
    except subprocess.TimeoutExpired:
        return {"game": config["game"], "agent_seat": config["agent_seat"],
                "status": "timeout", "error": f"Game worker exceeded {timeout} seconds"}
    if worker.returncode:
        return {"game": config["game"], "agent_seat": config["agent_seat"],
                "status": "worker_error", "error": worker.stderr[-8000:], "returncode": worker.returncode}
    try:
        return json.loads(worker.stdout)
    except json.JSONDecodeError:
        return {"game": config["game"], "agent_seat": config["agent_seat"],
                "status": "worker_error", "error": "Worker did not return JSON", "stdout": worker.stdout[-2000:]}


def wilson_interval(wins, total):
    if not total:
        return None
    z = 1.959963984540054
    p = wins / total
    denominator = 1 + z * z / total
    center = (p + z * z / (2 * total)) / denominator
    margin = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / denominator
    return [max(0, center - margin), min(1, center + margin)]


def summarize(games):
    completed = [g for g in games if g["status"] == "completed"]
    outcomes = Counter(g["outcome"] for g in completed)
    total = len(completed)
    errors = Counter(g["status"] for g in games if g["status"] != "completed")

    def split(key):
        bins = {}
        for g in completed:
            value = str(g["agent_seat"]) if key == "seat" else "first" if g["first_player"] == g["agent_seat"] else "second"
            bucket = bins.setdefault(value, {"games": 0, "wins": 0, "losses": 0, "draws": 0})
            bucket["games"] += 1
            bucket[{"win": "wins", "loss": "losses", "draw": "draws"}[g["outcome"]]] += 1
        return bins

    decisions = sum(g["decisions"][g["agent_seat"]] for g in completed)
    seconds = sum(g["policy_seconds"][g["agent_seat"]] for g in completed)
    return {"attempted": len(games), "completed": total,
            "wins": outcomes["win"], "losses": outcomes["loss"], "draws": outcomes["draw"],
            "errors": dict(errors), "win_rate": outcomes["win"] / total if total else None,
            "win_rate_95pct_wilson": wilson_interval(outcomes["win"], total),
            "by_seat": split("seat"), "by_start_order": split("order"),
            "agent_decisions": decisions,
            "agent_mean_decision_ms": 1000 * seconds / decisions if decisions else None,
            "agent_max_decision_ms": 1000 * max((g["max_policy_seconds"][g["agent_seat"]] for g in completed), default=0),
            "mean_game_seconds": sum(g["wall_seconds"] for g in completed) / total if total else None}


def fingerprint():
    """Identify the exact code and native library used for this run."""
    import importlib
    load_sdk()
    sim = importlib.import_module("cg.sim")
    files = [ROOT / "main.py", ROOT / "deck.csv", *sorted((ROOT / "pokemon_agent").glob("*.py")),
             *sorted((ROOT / "decks").glob("*.csv")), Path(sim.lib_path)]
    if (ROOT / "main_alakazam.py").is_file():
        files.append(ROOT / "main_alakazam.py")
    return {str(p.relative_to(ROOT)).replace("\\", "/"): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}


def save_report(path, report):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


def main():
    from .agents import AGENT_NAMES, default_deck
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--games", type=int, default=20)
    parser.add_argument("--agent", choices=AGENT_NAMES, default="abomasnow")
    parser.add_argument("--opponent", choices=AGENT_NAMES, default="random")
    parser.add_argument("--agent-deck", type=Path, default=None)
    parser.add_argument("--opponent-deck", type=Path, default=None)
    parser.add_argument("--seed", type=int, default=42, help="Policy RNG seed; native deals remain unseeded")
    parser.add_argument("--game-timeout", type=float, default=30)
    parser.add_argument("--decision-timeout", type=float, default=1)
    parser.add_argument("--max-decisions", type=int, default=5000)
    parser.add_argument("--output", type=Path, default=None)
    parser.add_argument("--trace-dir", type=Path, help="Optional per-decision visible observations as JSONL")
    parser.add_argument("--worker", action="store_true", help=argparse.SUPPRESS)
    args = parser.parse_args()
    if args.worker:
        print(json.dumps(play_game(json.load(sys.stdin)), ensure_ascii=False))
        return 0
    if args.games < 1 or args.max_decisions < 1 or args.game_timeout <= 0 or args.decision_timeout <= 0:
        parser.error("Game counts, decision counts and time limits must be positive")
    now = datetime.now(timezone.utc)
    output = args.output or ROOT / "results" / f"{now:%Y%m%dT%H%M%S%fZ}-{args.agent}-vs-{args.opponent}.json"
    if output.exists():
        parser.error(f"Refusing to overwrite existing results: {output}")
    deck = read_deck(args.agent_deck) if args.agent_deck else default_deck(args.agent)
    opponent_deck = (read_deck(args.opponent_deck) if args.opponent_deck else
                     list(deck) if args.opponent == "random" else default_deck(args.opponent))
    report = {"schema_version": 1, "started_utc": now.isoformat(), "run_status": "running",
              "agent": args.agent, "opponent": args.opponent, "requested_games": args.games,
              "agent_deck": deck, "opponent_deck": opponent_deck, "policy_seed": args.seed,
              "native_rng_seeded": False, "format": "independent single games, not BO3",
              "start_order": "Four-game cycle balances both seats and first/second; IS_FIRST is controlled by runner",
              "limits": {"game_timeout_seconds": args.game_timeout, "decision_timeout_seconds": args.decision_timeout,
                         "max_decisions": args.max_decisions},
              "python": sys.version, "platform": platform.platform(), "source_sha256": fingerprint(), "games": []}
    save_report(output, report)
    for number in range(args.games):
        seat = number % 2
        first = seat if (number // 2) % 2 == 0 else 1 - seat
        config = {"game": number + 1, "agent": args.agent, "opponent": args.opponent,
                  "agent_seat": seat, "first_player": first, "policy_seed": args.seed + 2 * number,
                  "agent_deck": deck, "opponent_deck": opponent_deck,
                  "max_decisions": args.max_decisions, "decision_timeout": args.decision_timeout,
                  "trace_file": str((args.trace_dir / f"game-{number + 1:04d}.jsonl").resolve()) if args.trace_dir else None}
        result = run_isolated(config, args.game_timeout)
        report["games"].append(result)
        report["summary"] = summarize(report["games"])
        save_report(output, report)
        print(f"{number + 1}/{args.games}: {result.get('outcome', result['status'])}, seat {seat}, "
              f"first {'agent' if seat == first else 'opponent'}", flush=True)
    report["run_status"] = "complete" if not report["summary"]["errors"] else "completed_with_errors"
    report["finished_utc"] = datetime.now(timezone.utc).isoformat()
    save_report(output, report)
    print(json.dumps(report["summary"], indent=2))
    print(f"Saved {output.resolve()}")
    return 1 if report["summary"]["errors"] else 0


if __name__ == "__main__":
    raise SystemExit(main())
''',
    'main.py': r'''"""Alternative entry point; the Alakazam notebook packages this as main.py."""
from functools import lru_cache
from pokemon_agent.alakazam import AlakazamAgent


@lru_cache(maxsize=1)
def _policy():
    return AlakazamAgent()


def agent(obs_dict):
    return _policy()(obs_dict)
''',
    'deck.csv': r'''741
741
741
741
742
742
742
742
743
743
743
743
65
65
65
65
66
66
66
66
5
5
5
5
19
19
19
19
1231
1231
1231
1231
1225
1225
1225
1225
1152
1152
1152
1152
1086
1086
1086
1086
1079
1079
1079
1079
1227
1227
1182
1182
1182
1184
1184
1097
1097
1264
1264
1413
''',
    'decks/abomasnow.csv': r'''1158
721
721
722
722
722
722
723
723
723
723
1145
1145
1145
1145
1205
1205
1227
1227
1227
1227
1235
1235
1235
1235
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
''',
    'decks/alakazam.csv': r'''741
741
741
741
742
742
742
742
743
743
743
743
65
65
65
65
66
66
66
66
5
5
5
5
19
19
19
19
1231
1231
1231
1231
1225
1225
1225
1225
1152
1152
1152
1152
1086
1086
1086
1086
1079
1079
1079
1079
1227
1227
1182
1182
1182
1184
1184
1097
1097
1264
1264
1413
''',
}

def prepare_workspace():
    for relative, source in PROJECT_SOURCES.items():
        destination = WORK_DIR / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(source, encoding='utf-8', newline='\n')

prepare_workspace()
print(f'Prepared {len(PROJECT_SOURCES)} agent files, including pokemon_agent/sdk.py')


## Evaluate the selected pilots and decks

Random uses the candidate deck; a named opponent uses its own deck. Failed games
are reported separately and never counted as wins. These are independent single
games, not Kaggle BO3 matches. Results include a confidence interval and seat/start-order
splits. The local game and decision deadlines below are safety guards, not
verified competition limits.


In [ ]:
if "prepare_workspace" not in globals():
    raise RuntimeError("Run the agent-file setup cell above first, or choose Run All.")
prepare_workspace()
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
REPORT_PATH = WORK_DIR / "results" / f"notebook-{stamp}.json"
command = [sys.executable, "-X", "utf8", "-m", "pokemon_agent.runner",
           "--agent", AGENT, "--opponent", OPPONENT,
           "--games", str(GAMES), "--seed", str(POLICY_SEED),
           "--game-timeout", "30", "--decision-timeout", "1",
           "--output", str(REPORT_PATH)]
subprocess.run(command, cwd=WORK_DIR, check=True)
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
print(json.dumps(report["summary"], indent=2))


## Build the Linux submission bundle

The archive places `main.py` and `deck.csv` at its root and includes the Python
modules and supplied x86-64 Linux engine. Building this file does not submit it.
Validate it in the actual Kaggle runtime before uploading; local Windows tests
do not establish Linux runtime compatibility.


In [ ]:
if "prepare_workspace" not in globals():
    raise RuntimeError("Run the agent-file setup cell above first, or choose Run All.")
prepare_workspace()
linux_library = WORK_DIR / "cg/libcg.so"
if not linux_library.is_file():
    raise FileNotFoundError("The supplied x86-64 Linux library cg/libcg.so is missing")
members = [WORK_DIR / "main.py", WORK_DIR / "deck.csv", linux_library]
members += sorted((WORK_DIR / "pokemon_agent").glob("*.py"))
members += sorted((WORK_DIR / "cg").glob("*.py"))
members += sorted((WORK_DIR / "decks").glob("*.csv"))
ARCHIVE_PATH = WORK_DIR / "submission.tar.gz"
with tarfile.open(ARCHIVE_PATH, "w:gz") as archive:
    for path in members:
        archive.add(path, arcname=path.relative_to(WORK_DIR).as_posix())
print(f"Submission bundle: {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 1024**2:.2f} MiB)")
print(f"Benchmark report: {REPORT_PATH}")
